# Phase 3 — Baseline 2: Tabular ML on Engineered Features

5-way stream ablation × 2 label policies × 5 LOYO folds = 50 fits.

**This notebook is RESUMABLE.** Each fold writes its results to disk immediately.
If the kernel dies or you interrupt, just re-run the same cells — already-done
folds are skipped automatically.

By default we use **HistGradientBoostingRegressor** (sklearn's fast LightGBM-equivalent),
which is 5–10× faster than RandomForest on this data and usually as accurate.
Switch `MODEL_TYPE = 'rf'` if you specifically want Random Forest.


## 1. Setup

In [7]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, os, time
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase3_outputs' / 'rf'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === KNOBS ===
YEARS         = [2017, 2018, 2019, 2020, 2021]
MODEL_TYPE    = 'rf'   # 'hgb' (fast) or 'rf' (classic RandomForest)
N_ESTIMATORS  = 200      # trees for RF, max_iter for HGB
STREAMS       = ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']
POLICIES      = ['all10', 'rank1']

print(f'Model type   : {MODEL_TYPE}')
print(f'N estimators : {N_ESTIMATORS}')
print(f'Streams      : {STREAMS}')
print(f'Policies     : {POLICIES}')
print(f'Output dir   : {OUTPUT_DIR}')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Model type   : rf
N estimators : 200
Streams      : ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']
Policies     : ['all10', 'rank1']
Output dir   : /fs1/home/atikul/Phenology/phase3_outputs/rf


## 2. Load bundles

In [10]:
from phase3_baselines.dataset import (load_all_bundles, loyo_split, SelectionPolicy,
                                       label_peak_days)
from phase3_baselines.engineered_features import compute_for_bundle
from phase3_baselines.baseline_rf import train_tabular, predict_models, STREAM_CONFIGS
from phase3_baselines.evaluate import evaluate_predictions
from phase3_baselines.resumable import append_row, already_done, save_pred_npz

bundles = load_all_bundles(TENSORS_DIR, years=YEARS)
phase_ids = next(iter(bundles.values()))['phase_ids']
print(f'Loaded years: {sorted(bundles.keys())}')
print(f'Phases: {phase_ids}')


Loaded years: [2017, 2018, 2019, 2020, 2021]
Phases: [10, 12, 15, 18, 19, 21, 24]


## 3. Run the 50-fold grid (resumable)

In [9]:
RESULTS_CSV = OUTPUT_DIR / f'rf_results_{MODEL_TYPE}.csv'
PREDS_NPZ   = OUTPUT_DIR / f'rf_predictions_{MODEL_TYPE}.npz'

policy_map = {
    'all10': SelectionPolicy(use_only_rank1=False),
    'rank1': SelectionPolicy(use_only_rank1=True),
}

# Resume support: skip (stream, policy, test_year) tuples already in the CSV
done = already_done(RESULTS_CSV, ['stream', 'policy', 'test_year'])
print(f'Already-done folds: {len(done)} (of {len(STREAMS)*len(POLICIES)*len(YEARS)})')

# Cache engineered features per (stream, policy, fold) within this session
feat_cache = {}

def get_features(bundle_dict, feats_subset, cache_key):
    if cache_key not in feat_cache:
        X, _ = compute_for_bundle(bundle_dict, restrict_features=feats_subset)
        feat_cache[cache_key] = X
    return feat_cache[cache_key]

total_folds = len(STREAMS) * len(POLICIES) * len(YEARS)
pbar = tqdm(total=total_folds, desc='RF folds')

for s in STREAMS:
    feats_subset = STREAM_CONFIGS[s]
    for pol_name in POLICIES:
        policy = policy_map[pol_name]
        for test_y in YEARS:
            key = (s, pol_name, int(test_y))
            if key in done:
                pbar.update(1); continue

            t0 = time.time()
            train_b, test_b = loyo_split(bundles, test_year=test_y, policy=policy)
            if len(train_b['xid']) == 0 or len(test_b['xid']) == 0:
                pbar.update(1); continue

            X_tr = get_features(train_b, feats_subset, ('tr', s, pol_name, test_y))
            X_te = get_features(test_b,  feats_subset, ('te', s, pol_name, test_y))
            Y_tr = label_peak_days(train_b['Y'], train_b['label_mask'])
            Y_te = label_peak_days(test_b['Y'],  test_b['label_mask']).astype(np.float32)

            models = train_tabular(X_tr, Y_tr,
                                    model_type=MODEL_TYPE,
                                    n_estimators=N_ESTIMATORS)
            pred = predict_models(models, X_te, n_phases=Y_te.shape[1])

            report = evaluate_predictions(
                pred_days=pred, true_days=Y_te,
                phase_ids=phase_ids,
                years=test_b['harvest_year'], ranks=test_b['rank'],
            )
            dt = time.time() - t0

            # Persist immediately
            for phase, m in report['per_phase'].iterrows():
                append_row(RESULTS_CSV, {
                    'stream': s, 'policy': pol_name, 'test_year': int(test_y),
                    'phase': int(phase), 'fit_secs': round(dt, 2),
                    **m.to_dict()
                })
            save_pred_npz(PREDS_NPZ, f'{s}__{pol_name}__{test_y}',
                          xid=test_b['xid'], bag=test_b['bag_id'],
                          rank=test_b['rank'], year=test_b['harvest_year'],
                          yt=Y_te, yp=pred)
            pbar.set_postfix({'last_fold': f'{s}/{pol_name}/{test_y}', 'secs': f'{dt:.1f}'})
            pbar.update(1)

pbar.close()
print('\nAll folds processed. Reload from CSV for analysis.')


Already-done folds: 0 (of 50)


RF folds:   0%|          | 0/50 [00:00<?, ?it/s]


All folds processed. Reload from CSV for analysis.


## 4. Summarize

In [11]:
results = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(results)} rows from {RESULTS_CSV}')

# Headline: macro over phases, mean across LOYO folds
headline = (results.groupby(['stream', 'policy'])
                   .agg(macro_mae=('mae', 'mean'),
                        macro_r2 =('r2',  'mean'),
                        macro_within_5d=('within_5d', 'mean'),
                        macro_rmse=('rmse', 'mean'))
                   .reset_index())
headline_path = OUTPUT_DIR / f'rf_headline_{MODEL_TYPE}.csv'
headline.to_csv(headline_path, index=False)
print('\n=== Headline (macro over phases, mean across LOYO folds) ===')
print(headline.round(3).to_string(index=False))


Loaded 350 rows from /fs1/home/atikul/Phenology/phase3_outputs/rf/rf_results_rf.csv

=== Headline (macro over phases, mean across LOYO folds) ===
  stream policy  macro_mae  macro_r2  macro_within_5d  macro_rmse
     All  all10      7.760     0.031            0.406       9.761
     All  rank1      7.774     0.053            0.406       9.821
Met_only  all10      7.921    -0.006            0.401       9.956
Met_only  rank1      7.933     0.023            0.399       9.979
   S1+S2  all10      7.919    -0.014            0.400       9.990
   S1+S2  rank1      8.007    -0.006            0.397      10.122
 S1_only  all10      8.033    -0.049            0.396      10.139
 S1_only  rank1      8.058    -0.020            0.392      10.175
 S2_only  all10      8.004    -0.031            0.393      10.074
 S2_only  rank1      8.117    -0.032            0.391      10.247


In [12]:
# Phase-resolved table (MAE per stream × phase, averaged across policies and folds)
table = (results.groupby(['stream', 'phase'])
                .agg(mae=('mae','mean'),
                     r2=('r2','mean'),
                     within_5d=('within_5d','mean'))
                .reset_index())
print('\n=== MAE by (stream, phase), averaged across policies & folds ===')
print(table.pivot(index='phase', columns='stream', values='mae').round(2).to_string())
print('\n=== R² by (stream, phase) ===')
print(table.pivot(index='phase', columns='stream', values='r2').round(2).to_string())
print('\n=== within±5d by (stream, phase) ===')
print(table.pivot(index='phase', columns='stream', values='within_5d').round(2).to_string())



=== MAE by (stream, phase), averaged across policies & folds ===
stream   All  Met_only  S1+S2  S1_only  S2_only
phase                                          
10      8.53      8.84   8.68     8.77     8.82
12      9.43      9.89   9.51     9.55     9.66
15      9.21      9.25   9.59     9.64     9.61
18      5.66      5.63   5.95     6.04     6.01
19      7.72      7.77   7.93     7.93     8.03
21      7.11      7.12   7.27     7.40     7.38
24      6.71      7.00   6.81     6.99     6.92

=== R² by (stream, phase) ===
stream   All  Met_only  S1+S2  S1_only  S2_only
phase                                          
10      0.10      0.04   0.07     0.05     0.04
12      0.11      0.04   0.09     0.08     0.07
15      0.01     -0.00  -0.07    -0.08    -0.07
18     -0.03     -0.02  -0.13    -0.16    -0.16
19      0.03      0.02  -0.02    -0.03    -0.05
21      0.06      0.06   0.01    -0.02    -0.02
24      0.01     -0.08  -0.01    -0.08    -0.04

=== within±5d by (stream, phase) ===
s

In [13]:
# Policy comparison: does Rank-1 beat All-10?
pol_compare = (results.groupby(['stream','policy'])['mae'].mean()
                       .unstack('policy'))
pol_compare['Δ (all10 − rank1)'] = pol_compare['all10'] - pol_compare['rank1']
print('\n=== Policy comparison: macro MAE ===')
print(pol_compare.round(2).to_string())



=== Policy comparison: macro MAE ===
policy    all10  rank1  Δ (all10 − rank1)
stream                                   
All        7.76   7.77              -0.01
Met_only   7.92   7.93              -0.01
S1+S2      7.92   8.01              -0.09
S1_only    8.03   8.06              -0.03
S2_only    8.00   8.12              -0.11


---\n**Done.** Move on to `03_lstm_baseline.ipynb`.